# Inference using LSDB-style API

In [1]:
import lsdb

In [2]:
from distributed import Client

N_WORKERS, THREADS_PER_WORKER = 1, 1
client = Client(n_workers=N_WORKERS, threads_per_worker=THREADS_PER_WORKER, processes=True)

## Open a HATS catalog with LSDB

Open a TESS catalog with LSDB and use LSDB's `map_partitions` to apply a function to each row in the catalog. In this example, we're applying a function that removes nested rows that contain NaNs.

In [3]:
from lsdb import ConeSearch

tess = lsdb.open_catalog(
    "https://data.lsdb.io/hats/tess/tess_lightcurve",
    columns=["ticid", "ra_obj", "dec_obj", "lightcurve"],
)

cone = ConeSearch(ra=180.0, dec=0.0, radius_arcsec=1 * 3600)
tess_cone = tess.search(cone)


def drop_nans(df):
    return df.dropna(subset="lightcurve.sap_flux").dropna(subset="lightcurve")


tess_filtered_nans = tess_cone.map_partitions(drop_nans)

In [4]:
import pandas as pd
from lsdb_ml.hyrax.config_utils import ModelConfig

# The fields returned by the model for prediction
prediction_meta = pd.DataFrame(
    {
        "pred": pd.Series(dtype="float32"),
    }
)

# ModelConfig provides the most important config parameters
# If the user needs more control, then they would use the LSDB-then-Hyrax approach.
# "Make the easy things easy and the hard things possible"
my_model_config = ModelConfig(
    model="HyraxLoopback",
    model_weights_file="./loopback_weights/fake_weights.pth",
    output_fields=prediction_meta,
    # The rest are optional fields
    hyrax_config_file=None,  # To provide a complex or pre-existing Hyrax config
    batch_size=64,
    primary_id_field="ticid",
    input_fields=["ra_obj", "dec_obj", "lightcurve_sap_flux"],
)

## Context manager approach

Here we don't necessarily need to create the client (and cluster?) before hand,
instead we rely on the `ml_client` context manager to handle client creation and
plugin registration automatically.

The only required parameter for the `ml_client` context manager is the hyrax
configuration in order to instantiate the model on the worker.

Optional kwargs are passed directly to the `Client` constructor inside the context
manager.

In [ ]:
from lsdb_ml.hyrax.client import ml_client
from lsdb_ml.hyrax.worker_plugin import predict

with ml_client(my_model_config):  # + optional kwargs passed to Client(...)
    tess_pred = predict(tess_filtered_nans)
    tess_pred.compute()

## Explicit client registration approach
This approach doesn't rely on a context manager and thus requires that the
Hyrax plugin be explicitly registered with the client before performing inference.
i.e. the user is responsible for creating the client and registering the plugin explicitly.

It also means that subsequent computation steps would be possible since the client
would not be closed out automatically.

In [ ]:
from lsdb_ml.hyrax.worker_plugin import predict, register_hyrax_plugin

register_hyrax_plugin(client, my_model_config)

tess_pred = predict(tess_filtered_nans)

# Potentially perform additional computation steps here...

tess_pred.compute()

Computing Catalog:   0%|          | 0/4 [00:00<?, ?it/s]

2026-10-29 10:08:48,442 ignite.distributed.auto.auto_dataloader INFO: Use data loader kwargs for dataset '<torch.utils.data.da': 
	{'sampler': None, 'batch_size': 64, 'collate_fn': <bound method CollationMixin.collate of Name: data (primary dataset)
  Dataset class: DataframeDataset
  Data location: {'ticid': 'int64[pyarrow]', 'ra_obj': 'double[pyarrow]', 'dec_obj': 'double[pyarrow]', 'lightcurve': {'cadenceno': 'int32[pyarrow]', 'mom_centr1': 'double[pyarrow]', 'mom_centr1_err': 'float[pyarrow]', 'mom_centr2': 'double[pyarrow]', 'mom_centr2_err': 'float[pyarrow]', 'pdcsap_flux': 'float[pyarrow]', 'pdcsap_flux_err': 'float[pyarrow]', 'pos_corr1': 'float[pyarrow]', 'pos_corr2': 'float[pyarrow]', 'psf_centr1': 'double[pyarrow]', 'psf_centr1_err': 'float[pyarrow]', 'psf_centr2': 'double[pyarrow]', 'psf_centr2_err': 'float[pyarrow]', 'quality': 'int32[pyarrow]', 'sap_bkg': 'float[pyarrow]', 'sap_bkg_err': 'float[pyarrow]', 'sap_flux': 'float[pyarrow]', 'sap_flux_err': 'float[pyarrow]', 'ti

ticid      ra_obj   dec_obj  \
_healpix_29                                            
1801390049747628151   35028418  180.231003 -0.731668   
1801407390937259735  902599667  179.592094 -0.728306   
1801413923461859294   35017228  179.486135 -0.667983   
1801437502909795554  289512775  180.117791 -0.103421   
1849437955645283823   35088390  180.806375 -0.354058   
1849477626264654295  289512755  180.092844 -0.039708   
1849478524753555753  376786323  180.059503  0.042249   
1897524140783988621   35019784  179.637327  0.038598   
1897526176329951224  337733952   179.56048   0.08292   
1897526176329951224  337733952   179.56048   0.08292   
1897527144708845140  337733971  179.536466  0.177815   
1897547248087973751   96197975  179.350102  0.113772   
1945557543028358483  289512725  179.966861  0.131074   
1945557924347894929  376758233   179.88894  0.164427   
1945603786772284903  262782272  179.730184  0.674598   
1945603972457643070  337750673  179.609921  0.642193   
1945605559467551911  262782294  179.729209  0.793713   
1945606287117976888  337734099  179.537291  0.747558   
1945607782731432936  337750607  179.641259  0.872143   
1945617641001766722  262782311  179.797226  0.871664   
1945619139859449367  262782313  179.776005  0.880915   
1945697768892195288  446677641  179.383496  0.780604   
1945700639041892336  337734119   179.49457  0.823025   
1945701286358226074  337734120  179.556563  0.832628   
1945701612772606460  337734133  179.600427  0.888519   

                                                            lightcurve  \
_healpix_29                                                              
1801390049747628151  [{cadenceno: 953356, mom_centr1: NaN, mom_cent...   
1801407390937259735  [{cadenceno: 953356, mom_centr1: NaN, mom_cent...   
1801413923461859294  [{cadenceno: 953356, mom_centr1: NaN, mom_cent...   
1801437502909795554  [{cadenceno: 953356, mom_centr1: NaN, mom_cent...   
1849437955645283823  [{cadenceno: 953356, mom_centr1: NaN, mom_cent...   
1849477626264654295  [{cadenceno: 953356, mom_centr1: NaN, mom_cent...   
1849478524753555753  [{cadenceno: 953356, mom_centr1: NaN, mom_cent...   
1897524140783988621  [{cadenceno: 953356, mom_centr1: NaN, mom_cent...   
1897526176329951224  [{cadenceno: 953356, mom_centr1: NaN, mom_cent...   
1897526176329951224  [{cadenceno: 1463561, mom_centr1: NaN, mom_cen...   
1897527144708845140  [{cadenceno: 1463561, mom_centr1: NaN, mom_cen...   
1897547248087973751  [{cadenceno: 953356, mom_centr1: NaN, mom_cent...   
1945557543028358483  [{cadenceno: 953356, mom_centr1: NaN, mom_cent...   
1945557924347894929  [{cadenceno: 953356, mom_centr1: NaN, mom_cent...   
1945603786772284903  [{cadenceno: 1463561, mom_centr1: NaN, mom_cen...   
1945603972457643070  [{cadenceno: 1463561, mom_centr1: NaN, mom_cen...   
1945605559467551911  [{cadenceno: 1463561, mom_centr1: NaN, mom_cen...   
1945606287117976888  [{cadenceno: 1463561, mom_centr1: NaN, mom_cen...   
1945607782731432936  [{cadenceno: 1463561, mom_centr1: NaN, mom_cen...   
1945617641001766722  [{cadenceno: 1463561, mom_centr1: NaN, mom_cen...   
1945619139859449367  [{cadenceno: 1463561, mom_centr1: NaN, mom_cen...   
1945697768892195288  [{cadenceno: 1463561, mom_centr1: NaN, mom_cen...   
1945700639041892336  [{cadenceno: 1463561, mom_centr1: NaN, mom_cen...   
1945701286358226074  [{cadenceno: 1463561, mom_centr1: NaN, mom_cen...   
1945701612772606460  [{cadenceno: 1463561, mom_centr1: NaN, mom_cen...   

                           pred  
_healpix_29                      
1801390049747628151  180.231003  
1801407390937259735  179.592087  
1801413923461859294  179.486130  
1801437502909795554  180.117798  
1849437955645283823  180.806381  
1849477626264654295  180.092850  
1849478524753555753  180.059509  
1897524140783988621  179.637329  
1897526176329951224  179.560486  
1897526176329951224  179.560486  
1897527144708845140  179.536469  
1897547248087973751  179.350098  
1945557543028358483  179.966858  
194

In [6]:
client.close()